# Chapter 9 — Synthetic FHIR Research Workflow (v2026)

> **LangChain 1.x / 2026 refresh.** Pinned versions, optional LangSmith tracing, and reproducible synthetic data. **Research-support only; synthetic/de-identified data; no clinical decisions.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2009.%20LangChain%20for%20Medicine%20and%20Healthcare/LC4LSH_Chapter_9_Synthetic_FHIR_Research_Workflow.ipynb)

Work with **synthetic FHIR-like resources** (Patient / Observation / Condition bundles): validate structure, build timelines, and run consistency & missingness checks. Fully local — no API key required.

**Learning objectives**
- Parse and validate FHIR-like JSON resources with Pydantic.
- Build a per-patient clinical timeline from Observations/Conditions.
- Run missingness and cross-resource consistency checks.

> **Runtime / cost / data.** Runs locally by default. Optional LLM cells are gated and can be skipped. **Synthetic / de-identified data only. Not for diagnosis, triage, treatment, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.**


## Environment setup

Standard preamble so every chapter notebook starts the same way.


### Secrets

Keys are read from Colab Secrets if available, else from a local `.env`. All optional — the notebook runs without them (LLM cells are skipped).


In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)


OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set:", bool(OPENAI_API_KEY))


### Install pinned dependencies

Pinned versions keep the notebook reproducible. See `UPDATE_2026.md`.


In [ ]:
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4" "pydantic>=2.5" "pandas" "matplotlib"


### Optional LangSmith tracing

Set `LANGCHAIN_API_KEY` to enable tracing of any LLM calls.


In [ ]:
import os

LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter9-synthetic-fhir-research"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")


## Synthetic FHIR-like bundle

A small in-memory bundle so the notebook is fully reproducible and uses only synthetic data.

In [ ]:
import json
from datetime import datetime

BUNDLE = {
    "resourceType": "Bundle",
    "type": "collection",
    "entry": [
        {"resource": {"resourceType": "Patient", "id": "p1", "gender": "female", "birthDate": "1962-04-11",
                      "name": [{"family": "Synth", "given": ["Anna"]}]}},
        {"resource": {"resourceType": "Observation", "id": "o1", "status": "final",
                      "code": {"coding": [{"system": "http://loinc.org", "code": "8480-6", "display": "Systolic BP"}]},
                      "subject": {"reference": "Patient/p1"},
                      "effectiveDateTime": "2024-01-10",
                      "valueQuantity": {"value": 148, "unit": "mmHg"}}},
        {"resource": {"resourceType": "Observation", "id": "o2", "status": "final",
                      "code": {"coding": [{"system": "http://loinc.org", "code": "14682-9", "display": "Creatinine"}]},
                      "subject": {"reference": "Patient/p1"},
                      "effectiveDateTime": "2024-01-10",
                      "valueQuantity": {"value": 1.6, "unit": "mg/dL"}}},
        {"resource": {"resourceType": "Condition", "id": "c1", "clinicalStatus": {"text": "active"},
                      "code": {"coding": [{"system": "http://hl7.org/fhir/sid/icd-10", "code": "I10", "display": "Essential hypertension"}]},
                      "subject": {"reference": "Patient/p1"},
                      "onsetDateTime": "2023-06-01"}},
        {"resource": {"resourceType": "Observation", "id": "o3", "status": "final",
                      "code": {"coding": [{"system": "http://loinc.org", "code": "8480-6"}]},
                      "subject": {"reference": "Patient/p999"},
                      "valueQuantity": {"value": 130, "unit": "mmHg"}}},
    ],
}
print("entries:", len(BUNDLE["entry"]))


### Validate resources with Pydantic

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field, ValidationError


class Coding(BaseModel):
    system: Optional[str] = None
    code: Optional[str] = None
    display: Optional[str] = None


class CodeableConcept(BaseModel):
    coding: list[Coding] = Field(default_factory=list)


class Reference(BaseModel):
    reference: str


class ValueQuantity(BaseModel):
    value: float
    unit: Optional[str] = None


class Resource(BaseModel):
    resourceType: str
    id: str
    status: Optional[str] = None
    code: Optional[CodeableConcept] = None
    subject: Optional[Reference] = None
    effectiveDateTime: Optional[str] = None
    onsetDateTime: Optional[str] = None
    valueQuantity: Optional[ValueQuantity] = None
    gender: Optional[str] = None
    birthDate: Optional[str] = None


valid, errors = [], []
for e in BUNDLE["entry"]:
    r = e["resource"]
    try:
        valid.append(Resource(**r))
    except ValidationError as ve:
        errors.append((r.get("id"), ve.errors()))

print("valid:", len(valid), "errors:", len(errors))
for rid, err in errors:
    print(rid, err)


### Per-patient timeline

Chronological clinical events per patient, sorted by date.

In [ ]:
from collections import defaultdict

timeline = defaultdict(list)
for r in valid:
    if r.subject is None:
        continue
    pid = r.subject.reference.split("/")[-1]
    date = r.effectiveDateTime or r.onsetDateTime or "unknown"
    label = r.resourceType
    if r.code and r.code.coding:
        label += ": " + (r.code.coding[0].display or r.code.coding[0].code or "?")
    if r.valueQuantity:
        label += f" = {r.valueQuantity.value} {r.valueQuantity.unit or ''}"
    timeline[pid].append((date, label))

for pid, events in timeline.items():
    print(f"--- Patient {pid} ---")
    for date, label in sorted(events):
        print(f"  {date}: {label}")


### Consistency & missingness checks

- **Dangling references**: subjects pointing to a Patient id that does not exist.
- **Missing dates**: Observations/Conditions without an effective/onset date.

In [ ]:
patient_ids = {r.id for r in valid if r.resourceType == "Patient"}
dangling = [r.id for r in valid if r.subject and r.subject.reference.split("/")[-1] not in patient_ids]
missing_date = [r.id for r in valid if r.resourceType in ("Observation", "Condition")
                and not (r.effectiveDateTime or r.onsetDateTime)]

print("known patients:", patient_ids)
print("dangling references:", dangling)
print("missing dates:", missing_date)

report = {
    "patients": len(patient_ids),
    "resources": len(valid),
    "dangling_references": dangling,
    "missing_dates": missing_date,
}
with open("fhir_consistency_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("report written -> fhir_consistency_report.json")


## Limitations & safety

- **Research-support only.** Not for diagnosis, triage, treatment recommendation, dosage, prescribing, final billing/coding, EHR writes, or patient messaging.
- **Synthetic / de-identified data only.** Real PHI requires governance, BAA-covered infrastructure, and access controls.
- Optional LLM outputs are **drafts for human review** and can hallucinate — always verify against source data.
- Any scoring/eligibility logic here is illustrative and must be validated by qualified clinicians before any real use.


In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Q1. Why validate FHIR resources with Pydantic instead of trusting the JSON?</summary>
FHIR JSON is loosely typed. Pydantic enforces required fields and types per resource, surfacing malformed resources before they corrupt downstream analysis.
</details>

<details><summary>Q2. What is a "dangling reference" and why does it matter?</summary>
An Observation/Condition whose `subject.reference` points to a Patient id not present in the bundle. It indicates incomplete export or linkage errors and must be resolved before cohort analysis.
</details>

<details><summary>Q3. Why sort timeline events by a normalized date?</summary>
Observations use `effectiveDateTime`, Conditions use `onsetDateTime`. Normalizing to one date field lets you order heterogeneous events into a single chronological history.
</details>

### Task A — Add a MedicationRequest resource
Extend the schema with a `MedicationRequest` resource (medication CodeableConcept + authoredOn date) and add it to the timeline.

### Task B — Reference-range flagging
Add optional `referenceRange` to Observations and flag any value outside its range in the report.

### Task C — Bundle statistics
Compute per-resourceType counts and per-patient event counts, and render them as a small pandas table.

### Task D — Export a patient summary
Write a one-patient JSON summary (demographics + sorted timeline + active conditions) and validate it round-trips through the Pydantic schema.
